In [ ]:
#

In [ ]:
# Projects
# Classification


from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier("This is a promt injection")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


[{'label': 'NEGATIVE', 'score': 0.9929879307746887}]

In [ ]:
from datasets import load_dataset

# 50,000 movie reviews (25k train, 25k test)
imdb = load_dataset("imdb")
print(imdb['train'][0])
# {'text': 'I loved this movie...', 'label': 1}  # 0=negative, 1=positive

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
# https://huggingface.co/datasets/stanfordnlp/sst2
# Movie reviews with binary sentiment
sst2 = load_dataset("glue", "sst2")
# ~67k training examples
print(sst2['train'][0])

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [ ]:
# Mini Toy Project
# Find the good classifier within hugging face less than 3b parameters
# do own research and provide findings, no bloated code just simple your
# simple stats and accuracy



# Mini Real Project
# https://huggingface.co/protectai/deberta-v3-base-prompt-injection
# Part #1 Direct Classify
# https://huggingface.co/datasets/xTRam1/safe-guard-prompt-injection


# Part #2 Harder Classify
# https://huggingface.co/datasets/reshabhs/SPML_Chatbot_Prompt_Injection


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
# !pip install -U transformers datasets peft accelerate evaluate

In [ ]:
# # Part 1:

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model
import evaluate

# 0) Metric
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# 1) Load tokenizer + base model (binary classifier: 0 = safe, 1 = injection)
tokenizer = AutoTokenizer.from_pretrained("ProtectAI/deberta-v3-base-prompt-injection")
model = AutoModelForSequenceClassification.from_pretrained(
    "ProtectAI/deberta-v3-base-prompt-injection", num_labels=2
)

# 2) Load first dataset
ds1 = load_dataset("xTRam1/safe-guard-prompt-injection")  # columns: "text", "label"

# 2a) Create validation split from the training set (e.g., 10%)
# If the dataset already has 'validation', you can skip this split and use ds1["validation"].
splits1 = ds1["train"].train_test_split(test_size=0.1, seed=42)
train_ds1 = splits1["train"]
val_ds1 = splits1["test"]
test_ds1 = ds1["test"]

# 3) Tokenize
def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

train_tok1 = train_ds1.map(preprocess, batched=True, remove_columns=["text"])
val_tok1   = val_ds1.map(preprocess,   batched=True, remove_columns=["text"])
test_tok1  = test_ds1.map(preprocess,  batched=True, remove_columns=["text"])

for d in (train_tok1, val_tok1, test_tok1):
    d = d.rename_column("label", "labels") if "label" in d.column_names else d
    d.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 4) Add LoRA adapters (typical DeBERTa v2/v3 attention proj names)
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["query_proj", "key_proj", "value_proj", "o_proj"],
)
model = get_peft_model(model, lora_cfg)

# 5) Train on first dataset (safe-guard-prompt-injection)
print("\n" + "="*60)
print("STAGE 1: Training on safe-guard-prompt-injection dataset")
print("="*60 + "\n")

args1 = TrainingArguments(
    output_dir="deberta-pi-lora-stage1",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    num_train_epochs=20,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",          # or evaluation_strategy if your version supports it
    save_strategy="epoch",
    load_best_model_at_end=True,    # automatically reload the best checkpoint
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=1,             # keep only the best model
    logging_steps=50,
    report_to="none",
)

trainer1 = Trainer(
    model=model,
    args=args1,
    train_dataset=train_tok1,
    eval_dataset=val_tok1,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)  # stop if no improvement for 3 evals
    ],
)

trainer1.train()

# Evaluate on the held-out test set for stage 1
test_metrics1 = trainer1.evaluate(test_tok1)
print("\nStage 1 Test metrics:", test_metrics1)

# Save stage 1 model
model.save_pretrained("deberta-pi-lora-stage1-final")
print("\nStage 1 model saved to: deberta-pi-lora-stage1-final")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

...

Map:   0%|          | 0/824 [00:00<?, ? examples/s]

Map:   0%|          | 0/2060 [00:00<?, ? examples/s]


STAGE 1: Training on safe-guard-prompt-injection dataset



Epoch,Training Loss,Validation Loss,Accuracy
1,0.089400,0.093525,0.979369
2,0.069900,0.088593,0.986650
3,0.032500,0.084771,0.983010
4,0.043200,0.085945,0.986650
5,0.042600,0.102264,0.984223



Stage 1 Test metrics: {'eval_loss': 0.09012763947248459, 'eval_accuracy': 0.9854368932038835, 'eval_runtime': 16.0598, 'eval_samples_per_second': 128.271, 'eval_steps_per_second': 4.047, 'epoch': 5.0}

Stage 1 model saved to: deberta-pi-lora-stage1-final


In [ ]:

# 6) Load and prepare second dataset (SPML_Chatbot_Prompt_Injection)
print("\n" + "="*60)
print("STAGE 2: Training on SPML_Chatbot_Prompt_Injection dataset")
print("="*60 + "\n")

ds2 = load_dataset("reshabhs/SPML_Chatbot_Prompt_Injection")

# Prepare second dataset - create validation split if needed
if "validation" in ds2:
    train_ds2 = ds2["train"]
    val_ds2 = ds2["validation"]
    test_ds2 = ds2["test"] if "test" in ds2 else val_ds2
else:
    splits2 = ds2["train"].train_test_split(test_size=0.1, seed=42)
    train_ds2 = splits2["train"]
    val_ds2 = splits2["test"]
    test_ds2 = ds2["test"] if "test" in ds2 else val_ds2

# Check dataset columns and create appropriate preprocessing function
print(f"Dataset 2 columns: {train_ds2.column_names}")

# Determine text and label columns for second dataset
# SPML dataset has: 'System Prompt', 'User Prompt', 'Prompt injection' (label), 'Degree', 'Source'
# We'll combine System Prompt and User Prompt as the text input
def preprocess_ds2(batch):
    # Combine system and user prompts
    if 'System Prompt' in batch and 'User Prompt' in batch:
        combined_text = [f"{sys} {usr}" for sys, usr in zip(batch['System Prompt'], batch['User Prompt'])]
    elif 'User Prompt' in batch:
        combined_text = batch['User Prompt']
    elif 'System Prompt' in batch:
        combined_text = batch['System Prompt']
    else:
        # Fallback to any text-like column
        text_col = None
        for possible_col in ['text', 'prompt', 'input', 'sentence', 'content']:
            if possible_col in batch:
                text_col = possible_col
                break
        if text_col is None:
            text_col = [col for col in batch.keys() if col not in ['label', 'Prompt injection']][0]
        combined_text = batch[text_col]

    return tokenizer(combined_text, truncation=True, padding="max_length", max_length=256)

# First rename the label column, then tokenize
# 'Prompt injection' column contains the labels
if 'Prompt injection' in train_ds2.column_names:
    train_ds2 = train_ds2.rename_column("Prompt injection", "label")
    val_ds2 = val_ds2.rename_column("Prompt injection", "label")
    test_ds2 = test_ds2.rename_column("Prompt injection", "label")

# Tokenize second dataset - keep the label column
train_tok2 = train_ds2.map(preprocess_ds2, batched=True, remove_columns=[col for col in train_ds2.column_names if col != "label"])
val_tok2   = val_ds2.map(preprocess_ds2,   batched=True, remove_columns=[col for col in val_ds2.column_names if col != "label"])
test_tok2  = test_ds2.map(preprocess_ds2,  batched=True, remove_columns=[col for col in test_ds2.column_names if col != "label"])

# Rename label to labels and set format
train_tok2 = train_tok2.rename_column("label", "labels")
val_tok2 = val_tok2.rename_column("label", "labels")
test_tok2 = test_tok2.rename_column("label", "labels")

train_tok2.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_tok2.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_tok2.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 7) Continue training on second dataset
args2 = TrainingArguments(
    output_dir="deberta-pi-lora-stage2",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    num_train_epochs=20,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
    report_to="none",
)

trainer2 = Trainer(
    model=model,  # Continue training with the same model from stage 1
    args=args2,
    train_dataset=train_tok2,
    eval_dataset=val_tok2,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)
    ],
)

trainer2.train()

# Evaluate on the held-out test set for stage 2
test_metrics2 = trainer2.evaluate(test_tok2)
print("\nStage 2 Test metrics:", test_metrics2)

# 8) Save final model after both stages
print("\n" + "="*60)
print("Saving final model after both training stages")
print("="*60 + "\n")

# Save only the LoRA adapter
model.save_pretrained("deberta-pi-lora-final-adapter")
print("Final LoRA adapter saved to: deberta-pi-lora-final-adapter")

# Save full model if desired
trainer2.save_model("deberta-pi-lora-final-full")
print("Final full model saved to: deberta-pi-lora-final-full")

best_model_path = trainer2.state.best_model_checkpoint
print("Best model checkpoint from stage 2:", best_model_path)

print("\n" + "="*60)
print("Training complete!")
print(f"Stage 1 (safe-guard) test metrics: {test_metrics1}")
print(f"Stage 2 (SPML) test metrics: {test_metrics2}")
print("="*60)


STAGE 2: Training on SPML_Chatbot_Prompt_Injection dataset

Dataset 2 columns: ['System Prompt', 'User Prompt', 'Prompt injection', 'Degree', 'Source']


Map:   0%|          | 0/14410 [00:00<?, ? examples/s]

Map:   0%|          | 0/1602 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.216400,0.205629,0.941948
2,0.144300,0.217205,0.953184
3,0.149000,0.201847,0.955680
4,0.162000,0.158276,0.958801
5,0.106800,0.155212,0.957553
6,0.114200,0.157322,0.958801
7,0.096400,0.170767,0.959426
8,0.109800,0.139020,0.960674
9,0.104600,0.154524,0.960050
10,0.083700,0.149143,0.960050



Stage 2 Test metrics: {'eval_loss': 0.13901996612548828, 'eval_accuracy': 0.9606741573033708, 'eval_runtime': 12.2349, 'eval_samples_per_second': 130.937, 'eval_steps_per_second': 4.168, 'epoch': 10.0}

Saving final model after both training stages

Final LoRA adapter saved to: deberta-pi-lora-final-adapter
Final full model saved to: deberta-pi-lora-final-full
Best model checkpoint from stage 2: deberta-pi-lora-stage2/checkpoint-7208

Training complete!
Stage 1 (safe-guard) test metrics: {'eval_loss': 0.09012763947248459, 'eval_accuracy': 0.9854368932038835, 'eval_runtime': 16.0598, 'eval_samples_per_second': 128.271, 'eval_steps_per_second': 4.047, 'epoch': 5.0}
Stage 2 (SPML) test metrics: {'eval_loss': 0.13901996612548828, 'eval_accuracy': 0.9606741573033708, 'eval_runtime': 12.2349, 'eval_samples_per_second': 130.937, 'eval_steps_per_second': 4.168, 'epoch': 10.0}


In [ ]:
# ls
!zip -r trained-models.zip trained-models

  adding: trained-models/ (stored 0%)
  adding: trained-models/deberta-pi-lora-final-full/ (stored 0%)
  adding: trained-models/deberta-pi-lora-final-full/README.md (deflated 66%)
  adding: trained-models/deberta-pi-lora-final-full/training_args.bin (deflated 53%)
  adding: trained-models/deberta-pi-lora-final-full/adapter_model.safetensors (deflated 7%)
  adding: trained-models/deberta-pi-lora-final-full/adapter_config.json (deflated 56%)
  adding: trained-models/deberta-pi-lora-final-adapter/ (stored 0%)
  adding: trained-models/deberta-pi-lora-final-adapter/README.md (deflated 66%)
  adding: trained-models/deberta-pi-lora-final-adapter/adapter_model.safetensors (deflated 7%)
  adding: trained-models/deberta-pi-lora-final-adapter/adapter_config.json (deflated 56%)
  adding: trained-models/deberta-pi-lora-stage1-final/ (stored 0%)
  adding: trained-models/deberta-pi-lora-stage1-final/README.md (deflated 66%)
  adding: trained-models/deberta-pi-lora-stage1-final/adapter_model.safetenso

To zip a folder named `my_folder` into a file named `my_archive.zip`, you can use the following command:

In [ ]:
# Mini Toy Project
# Find the good classifier within hugging face less than 3b parameters
# do own research and provide findings, no bloated code just simple your
# simple stats and accuracy



# Mini Real Project
# https://huggingface.co/protectai/deberta-v3-base-prompt-injection
# Part #1 Direct Classify
# https://huggingface.co/datasets/xTRam1/safe-guard-prompt-injection


# Part #2 Harder Classify
# https://huggingface.co/datasets/reshabhs/SPML_Chatbot_Prompt_Injection

In [ ]:
# Mini Toy Project
# Find the good classifier within hugging face less than 3b parameters
# do own research and provide findings, no bloated code just simple your
# simple stats and accuracy



# Mini Real Project
# https://huggingface.co/protectai/deberta-v3-base-prompt-injection
# Part #1 Direct Classify
# https://huggingface.co/datasets/xTRam1/safe-guard-prompt-injection


# Part #2 Harder Classify
# https://huggingface.co/datasets/reshabhs/SPML_Chatbot_Prompt_Injection

In [ ]:
# Another Level
#  Part # 3 Nvidia does the job
# # Dataset https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-2.0
# # Model https://huggingface.co/nvidia/llama-3.1-nemoguard-8b-content-safety

# make the below code work till Thanksgiving
# and test on dataset and report stats


In [ ]:
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # Load base once
# base_model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Meta-Llama-3.1-8B-Instruct",
#     device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

# # Scenario 1: Content Safety Check
# print("=== Safety Check ===")
# safety_model = PeftModel.from_pretrained(
#     base_model,
#     "nvidia/llama-3.1-nemoguard-8b-content-safety"
# )

# prompt = "Is this content safe: [user input here]"
# inputs = tokenizer(prompt, return_tensors="pt").to(safety_model.device)
# outputs = safety_model.generate(**inputs, max_new_tokens=50)
# print(tokenizer.decode(outputs[0]))